In [ ]:
import os
import gzip
import pandas as pd
import matplotlib.pyplot as plt
import bar_chart_race as bcr
from collections import defaultdict
from datetime import datetime

In [ ]:
# --- 1. DATA LOADING AND PARSING ---

def parse_data(data_dir="data/pdb"):
    # Dictionaries for our data
    pdb_lengths = defaultdict(int)
    pdb_data = {}
    
    # 1a. Parsing pdb_seqres.txt (collecting only proteins and their lengths)
    seqres_path = os.path.join(data_dir, "pdb_seqres.txt")
    with open(seqres_path, 'r') as f:
        for line in f:
            if line.startswith(">"):
                # Example line: >101m_A mol:protein length:154  MYOGLOBIN
                if "mol:protein" in line:
                    parts = line.split()
                    pdb_id = parts[0][1:5].upper() # Extract 4-character ID, e.g., 101M
                    length_str = [p for p in parts if p.startswith("length:")][0]
                    length = int(length_str.split(":")[1])
                    pdb_lengths[pdb_id] += length # Sum chain lengths for a given PDB
    
    # 1b. Parsing entries.idx (extracting date, resolution, experiment, and name/header)
    idx_path = os.path.join(data_dir, "entries.idx")
    with open(idx_path, 'r') as f:
        # Skip the first 2 header lines
        next(f)
        next(f)
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 8:
                pdb_id = parts[0].upper()
                
                # We only care about structures containing proteins (found in seqres)
                if pdb_id not in pdb_lengths:
                    continue
                    
                header_name = parts[1]
                date_str = parts[2]
                
                # Convert MM/DD/YY date to year
                try:
                    date_obj = datetime.strptime(date_str, "%m/%d/%y")
                    # Correction for year (strptime for '94' might return 2094)
                    year = date_obj.year
                    if year > 2025:
                        year -= 100
                except:
                    continue
                
                resolution_str = parts[6]
                experiment = parts[7].upper()
                
                # Determine resolution (r)
                if "NMR" in experiment:
                    r = 2.5 # According to instructions for NMR r=2.5
                else:
                    try:
                        r = float(resolution_str)
                    except ValueError:
                        continue # Skip if resolution is missing and it's not NMR
                
                # Calculate value: n * (1/r)
                n = pdb_lengths[pdb_id]
                value = n * (1.0 / r)
                
                pdb_data[pdb_id] = {
                    'year': year,
                    'experiment': experiment,
                    'value': value,
                    'name': header_name
                }
                
    return pd.DataFrame.from_dict(pdb_data, orient='index')

In [ ]:
# --- 2. STATIC PIE CHART AND CLASSIFICATION ---

def create_pie_chart(df, class_mapping, **kwargs):
    """
    Creates a pie chart. 
    class_mapping: dictionary { 'Category Name': ['Method1', 'Method2'] }
    """
    mapped_data = defaultdict(float)
    skipped_elements = []
    
    # Iterate through all records
    for pdb_id, row in df.iterrows():
        exp = row['experiment']
        matched = False
        
        # Look for a match in defined classes
        for category, methods in class_mapping.items():
            if any(method in exp for method in methods):
                mapped_data[category] += row['value']
                matched = True
                break
                
        # If no match found, add to skipped elements
        if not matched:
            skipped_elements.append(f"{exp} ({row['name']})")
            
    # Drawing the pie chart
    plt.figure(figsize=(10, 8))
    plt.pie(mapped_data.values(), labels=mapped_data.keys(), autopct='%1.1f%%', startangle=140)
    plt.title('Share of experimental methods in protein structures in PDB $\sum(n \cdot 1/r)$')
    plt.axis('equal')
    plt.savefig('pie_chart_methods.png')
    plt.close()
    
    # Printing skipped elements (unique entries to avoid flooding the console)
    print("\n--- SKIPPED ELEMENTS ---")
    unique_skipped = list(set(skipped_elements))
    for skipped in unique_skipped[:20]: # Show first 20 unique for clarity
        print(f"- {skipped}")
    if len(unique_skipped) > 20:
        print(f"... and {len(unique_skipped) - 20} others.")
        
    return mapped_data

# --- 3. ANIMATED BARPLOTS (2005 - 2025) ---

def create_animated_barplots(df, class_mapping):
    # Mapping to main categories for the entire DataFrame
    def map_exp(exp):
        for category, methods in class_mapping.items():
            if any(method in exp for method in methods):
                return category
        return "OTHER"
        
    df['category'] = df['experiment'].apply(map_exp)
    
    # Filter years 2005-2025 and remove "OTHER"
    df_filtered = df[(df['year'] >= 2005) & (df['year'] <= 2025) & (df['category'] != "OTHER")]
    
    # Group by year and category, then calculate cumulative sum
    # PDB data volume grows over time, so we use accumulation
    yearly_data = df_filtered.groupby(['year', 'category'])['value'].sum().unstack(fill_value=0)
    yearly_data_cumsum = yearly_data.cumsum()
    
    # Version 1: Absolute numbers
    bcr.bar_chart_race(
        df=yearly_data_cumsum,
        filename='absolute_growth_2005_2025.mp4',
        orientation='h',
        sort='desc',
        n_bars=3,
        title='Data growth in PDB (absolute values $\sum(n \cdot 1/r)$)',
        steps_per_period=10,
        period_length=1000
    )
    
    # Version 2: Percentages (calculating percentage share for each year)
    yearly_data_pct = yearly_data_cumsum.div(yearly_data_cumsum.sum(axis=1), axis=0) * 100
    
    bcr.bar_chart_race(
        df=yearly_data_pct,
        filename='percentage_growth_2005_2025.mp4',
        orientation='h',
        sort='desc',
        n_bars=3,
        title='Percentage share of methods in PDB (2005-2025)',
        steps_per_period=10,
        period_length=1000
    )

In [ ]:
df = parse_data(data_dir="data/pdb")